# Ruh training repair validation

Private bounded validation of the real sentence/dialogue pipeline. This **3.58M pilot** is separate from the existing **46M production Ruh**. No private checkpoint is attached. No candidate is approved for production.

Public data: OpenAssistant/oasst1 (Apache-2.0) and Arabic Wikipedia (CC-BY-SA-3.0/GFDL). Exact revisions and attribution are saved in sources.json. GPU session limit900 seconds; training limit480 seconds; paid capacity is not used.


In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import torch
SESSION_STARTED = time.monotonic()
assert torch.cuda.is_available(), "Kaggle GPU accelerator is required"
print({"gpu": torch.cuda.get_device_name(0), "gpu_count": torch.cuda.device_count(), "torch": torch.__version__})
# Use Kaggle's existing Torch. Only small corpus dependencies are installed.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub==1.33.0", "pyarrow==25.0.1"], check=True, timeout=120)
REVISION = "aaf2e9e5cfab389448214da0a19cf4c28a534df3"
REPOSITORY = Path("/kaggle/working/mizan")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/CodeWithJuber/mizan.git", str(REPOSITORY)], check=True, timeout=90)
subprocess.run(["git", "fetch", "--depth=1", "origin", REVISION], cwd=REPOSITORY, check=True, timeout=90)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPOSITORY, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPOSITORY, text=True).strip()
assert actual == REVISION
os.environ["HF_HOME"] = "/kaggle/working/.hf-cache"
print({"training_revision": actual, "setup_seconds": time.monotonic() - SESSION_STARTED})


In [ ]:
# All commands preserve exit codes; no shell pipelines or tail masking.
subprocess.run([sys.executable, "-m", "scripts.prepare_sequence_data", "--output-dir", "/kaggle/working/data", "--dialogues", "1500", "--sentences", "500"], cwd=REPOSITORY, check=True, timeout=150)


In [ ]:
remaining = 820 - (time.monotonic() - SESSION_STARTED)
assert remaining > 100, "Insufficient session budget after data/setup"
training_budget = min(480, int(remaining - 70))
subprocess.run([sys.executable, "-m", "ruh_model.train_sequence", "--data", "/kaggle/working/data/training.jsonl", "--source-metadata", "/kaggle/working/data/sources.json", "--output", "/kaggle/working/validation-run", "--device", "cuda", "--steps", "1000", "--max-seconds", str(training_budget), "--seq-len", "384", "--batch-size", "2", "--d-model", "256", "--layers", "4", "--lr", "0.0003"], cwd=REPOSITORY, check=True, timeout=int(remaining))


In [ ]:
import json, hashlib, zipfile
run = Path("/kaggle/working/validation-run")
report = json.loads((run / "evaluation.json").read_text())
report["training_revision"] = REVISION
report["session_seconds"] = time.monotonic() - SESSION_STARTED
report["accelerator"] = torch.cuda.get_device_name(0)
assert report["tokenizer_version"] == 2
assert report["promotion"]["approved"] is False
(run / "evaluation.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps({name: report[name] for name in ["status", "parameters", "held_out_before", "held_out_after", "generations", "promotion"]}, ensure_ascii=False, indent=2))
# Only candidate outputs; private input datasets and credentials are never packaged.
with zipfile.ZipFile("/kaggle/working/validation-candidate.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(run.rglob("*")):
        if file.is_file():
            archive.write(file, arcname=str(file.relative_to(run.parent)))
print({"session_seconds": time.monotonic() - SESSION_STARTED, "promotion_approved": False})
